# Notebook 01: Where Baseline RAG Fails

Testing baseline RAG against real-world customer queries to identify limitations.

## 1. Setup

We use two clients:
- **OpenAI client** - For the LLM (gpt-4o)
- **bedrock-agent-runtime** - For Knowledge Base retrieval via `retrieve()` API

In [15]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import boto3

load_dotenv()

LLM_BASE_URL = os.getenv('LLM_BASE_URL')
LLM_API_KEY = os.getenv('LLM_API_KEY')
LLM_MODEL = os.getenv('LLM_MODEL', 'gpt-4o')
KNOWLEDGE_BASE_ID = os.getenv('KNOWLEDGE_BASE_ID')

llm_client = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY)
bedrock_agent = boto3.client('bedrock-agent-runtime', region_name='us-east-1')

print('[OK] Configuration loaded')
print(f'  LLM Model: {LLM_MODEL}')
print(f'  Knowledge Base: {KNOWLEDGE_BASE_ID}')

[OK] Configuration loaded
  LLM Model: gpt-4o
  Knowledge Base: HAMKFMPMEM


## 2. Baseline RAG with bedrock_agent.retrieve()

The `retrieve()` API:
- Takes a query, converts it to an embedding
- Searches the Knowledge Base for similar document chunks
- Returns the top N most relevant chunks

```
retrieve(
    knowledgeBaseId='...',           # Your KB ID
    retrievalQuery={'text': query},  # User's question  
    retrievalConfiguration={
        'vectorSearchConfiguration': {'numberOfResults': 3}  # How many chunks
    }
)
```

This is the same RAG pattern from Part 4, but Bedrock manages the vector store.

In [16]:
def baseline_rag(query: str) -> str:
    """Simple baseline RAG: Retrieve from KB + Generate with LLM"""
    
    # Retrieve from Knowledge Base
    response = bedrock_agent.retrieve(
        knowledgeBaseId=KNOWLEDGE_BASE_ID,
        retrievalQuery={'text': query},
        retrievalConfiguration={'vectorSearchConfiguration': {'numberOfResults': 3}}
    )
    
    # Build context
    context_parts = []
    for idx, result in enumerate(response.get('retrievalResults', []), 1):
        context_parts.append(f"[Doc {idx}] {result['content']['text']}")
    context = "\n\n".join(context_parts)
    
    # Generate answer
    llm_response = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful e-commerce assistant. Answer based on the provided context."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}
        ],
        temperature=0.1
    )
    
    return llm_response.choices[0].message.content

In [17]:
# Test with a policy question - this works fine
print("Testing Baseline RAG - Policy Question (This Works)")
print("=" * 60)
answer = baseline_rag("What is your return policy?")
print(f"Q: What is your return policy?")
print(f"A: {answer}")

Testing Baseline RAG - Policy Question (This Works)
Q: What is your return policy?
A: Our return policy allows you to return products within 30 days of delivery. Items must be in their original condition with all tags attached to be eligible for a return. Damaged or defective items qualify for a full refund with no restocking fee. However, for returns due to a change of mind, a 15% restocking fee will be applied to the refund amount. Please refer to our Returns page for detailed instructions.


## 3. Failure Scenarios

The `retrieve()` API only searches indexed documents. It cannot:
- Query databases (orders, inventory)
- Call external APIs (weather)
- Take actions (create returns)

Let's see it fail on real customer queries.

In [18]:
# FAILURE 1: Order Status Lookup
# Problem: No access to order database

print("=" * 70)
print("FAILURE 1: Order Status Lookup")
print("=" * 70)

query = "What is the status of my order #ORD-12345?"
print(f"\nCustomer asks: {query}\n")
print("Baseline RAG Response:")
print("-" * 50)
print(baseline_rag(query))
print("-" * 50)
print("\nProblem: Returns generic instructions, not actual order status.")

FAILURE 1: Order Status Lookup

Customer asks: What is the status of my order #ORD-12345?

Baseline RAG Response:
--------------------------------------------------
I'm sorry, but I don't have access to specific order details or statuses. To check the status of your order #ORD-12345, please log into your account and navigate to the 'Order History' section, where you can find the tracking information. Alternatively, you can use the tracking number sent to your email to track your order on our website. If you need further assistance, consider reaching out to customer support.
--------------------------------------------------

Problem: Returns generic instructions, not actual order status.


In [19]:
# FAILURE 2: Weather-Based Shipping Delay
# Problem: No access to weather APIs

print("=" * 70)
print("FAILURE 2: Weather-Based Shipping Delay")
print("=" * 70)

query = "Will my package to Miami be delayed? I heard there's a hurricane."
print(f"\nCustomer asks: {query}\n")
print("Baseline RAG Response:")
print("-" * 50)
print(baseline_rag(query))
print("-" * 50)
print("\nProblem: Cannot check real-time weather or predict actual delays.")

FAILURE 2: Weather-Based Shipping Delay

Customer asks: Will my package to Miami be delayed? I heard there's a hurricane.

Baseline RAG Response:
--------------------------------------------------
I don't have specific information about current weather conditions or their impact on shipping. However, if there is a hurricane affecting Miami, it is possible that your package could be delayed. For the most accurate information, I recommend contacting the shipping carrier or our customer support team to inquire about any potential delays due to weather conditions.
--------------------------------------------------

Problem: Cannot check real-time weather or predict actual delays.


In [20]:
# FAILURE 3: Real-Time Inventory Check
# Problem: No access to inventory system

print("=" * 70)
print("FAILURE 3: Real-Time Inventory Check")
print("=" * 70)

query = "Do you have the blue iPhone 15 Pro case in stock?"
print(f"\nCustomer asks: {query}\n")
print("Baseline RAG Response:")
print("-" * 50)
print(baseline_rag(query))
print("-" * 50)
print("\nProblem: Cannot check current stock levels.")

FAILURE 3: Real-Time Inventory Check

Customer asks: Do you have the blue iPhone 15 Pro case in stock?

Baseline RAG Response:
--------------------------------------------------
I'm unable to check the current stock status of specific products like the blue iPhone 15 Pro case. If it's not available, it may be temporarily out of stock. I recommend checking back later or signing up for notifications if that option is available.
--------------------------------------------------

Problem: Cannot check current stock levels.


In [21]:
# FAILURE 4: Create a Return Request
# Problem: Cannot take actions

print("=" * 70)
print("FAILURE 4: Create a Return Request")
print("=" * 70)

query = "I want to return order #ORD-67890. The product was defective. Please process the return."
print(f"\nCustomer asks: {query}\n")
print("Baseline RAG Response:")
print("-" * 50)
print(baseline_rag(query))
print("-" * 50)
print("\nProblem: Gives instructions instead of actually creating the return.")

FAILURE 4: Create a Return Request

Customer asks: I want to return order #ORD-67890. The product was defective. Please process the return.

Baseline RAG Response:
--------------------------------------------------
To return your defective product from order #ORD-67890, please follow these steps:

1. Log into your account on our website.
2. Navigate to your account dashboard and locate the order #ORD-67890.
3. Submit a return request for the defective product, including any necessary details about the defect.
4. Once your return request is approved, you will receive a return shipping label via email within 24 hours.
5. Print the label and attach it to the package.
6. Ship the item back to us within 5 business days of receiving the label.

If you encounter any issues or need further assistance, please contact our customer support team.
--------------------------------------------------

Problem: Gives instructions instead of actually creating the return.


## 4. Summary

**Baseline RAG Limitations:**

| Scenario | Customer Expects | RAG Gives |
|----------|-----------------|----------|
| Order status | Actual order info | Generic instructions |
| Weather delays | Real-time prediction | Standard shipping times |
| Inventory | Stock availability | "Check our website" |
| Create return | Actually process it | Instructions |

**Root Cause:** No tools - RAG can only search documents.

**Solution:** Agentic RAG = RAG + Tools + Reasoning (ReAct pattern)

**Next:** Build tools to solve these failures.